## **Project Overeiew**
### We are going to create a team of AI  Agents that can collaborate and interact with humans using Autogen framework.
### Imagine you have AI assistants that don't just answer your questions but can take on roles like a marketing manager, communicate with each other, and work towards a common goal.
### What if the assistant could even use different AI "brains" (like GPT for one and Gemini for another) within the same team? This is the flexibility we will explore.
### Autogen is the framework that we are going to use.
### -We are going to have Cheif Marketing Officer (CMO) powered by Google Gemini.
### -We are also going to have marketing manager powered by OpenAI  API.
### -We are going to have Social Media Strategist powered by Claude (Antropic),
### -We are going have human in the loop.
### -We are also going have a group Chat Manager that is managing who to speak and when.
### All of these are going to work together.
### So an AI working with human to brainstorm a marketing campaign.


# **Key Learning Objectives**
### 1. Understand the basic concept of multi-model AI agents in AutoGen.
### 2. Configure AutoGen and load API keys for both OpenAI and Google Gemini.
### 3. Create single model AI Agents including a marketing CMO and Brand Market using OpenAI's GPT first.
### 4. Make the AI agents discuss ideas using AutoGen's chat capabilities.
### 5. Modify agents to use different multi-model LLMs (e.g, Gemini for CMO, GPT for Marketer).
### 6. Introduce Human Oversight: Add a "user proxy" agent with a human in the loop.



## **What is an AutoGen?**
### -In AutoGen, an agent is an entity that can send and receive messages to and from other agents in its environment.
### -An agent can be powered by LLMs models, code executors (such as an IPython kernel), human and customizable components.


# **ConversableAgent**
### -It can be powered by LLM.
### -It can get access to tools.
### -It can also have human in the loop.
### -It can execute code.


## **Group Chat**
### -AutoGen provides a conversation pattern called "group chat", which involves more than 2 agents.
### -The idea of group chat is that all agents contribute to a single conversation thread and share the same context. This is useful for tasks that require collaboration among multiple agents.
### -A group chat is orchestrated by a special agent type "GroupChatManager". At first, the GroupChatManager selects an agent to speak. Then the selected agent speaks, and the message is sent back to the GroupChatManager, who broadcasts the message to all other agents in the group. This process repeats until the conversation stops.

## **Practice Opportunity**
### -Imagine using AI agent to plan a vacation.
### -What are 2-3 distinct roles you could assign?
### -Describe what each agent would specialize, in and how they would interact with each other.
### -Think about how you would design their system messages to guide their behaviour.

## **Here are three AI agents' roles for vacation planning:**
### 1. Destination Research Agent: This specializes in matching traveller preferences with suitable locations. This agent would have extensive knowledge of global destinations, seasonal considerations, and travel trends. It could suggest destinationsbased on budget, interests, (advernture, relaxation, culture), weather preferences, and travel time constraints.
### 2. Logistics & Booking Agent: This focuses on finding and securing the best travel arrangements. This agent would search for flight options, accommodation deals, local transportation, and activity reservations. It would handle schedulling, price comparisons, and booking confirmations while ensuring all components work together seamlessly.
### 3. Local Experience Agent: It provides knowledge about the chosen destination. This agent would recommend authentic restaurants, hidden attractions, and cultural norms. It would create personalized daily itineraries based on travellers' interests, mobility needs, and pace preferences, while adapting to weather changes or unexpected events.

### **Create An AI Agent with Similar LLM (OpenAI) first**

## **Analogy: A Specialized Assistant**
### Imagine you have two assistants for a project:
### -Assistant A (The Planner): Their job is to understand the overall deal, break it down into steps, and delegate tasks. They focus on the big picture.
### -Assistant B (The Doer): Their job is to receive specific tasks given to them by the Planner. They focus on implementation.

### You wouldn't just give the Doer the final goal, you would have the Planner instruct the Doer. They might talk back and forth.

### Autogen helps us create these assistants and manage their conversations, regardless of the underlying AI model powering them.



### Let's start by creating the Chief Marketing Officer (CMO) and Brand Marketer agents, initially having both use the same LLM (OpenAI's GPT) for simplicity. This helps to understand the basic agent creation process.

In [ ]:
!pip install -q pyautogen openai python-dotenv gradio google-generativeai "ag2[gemini]"

In [ ]:
# Programmatically install to ensure it goes directly into the active Python 3.13 kernel
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyautogen", "autogen", "openai", "python-dotenv", "gradio", "google-generativeai", "ag2[gemini]"])

# Force refresh the system paths and invalidate the caches
import site
import importlib
for site_dir in site.getsitepackages():
    if site_dir not in sys.path:
        sys.path.append(site_dir)
importlib.invalidate_caches()

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Ignore Necessary Libraries
import os
import autogen
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import random
import google.generativeai as genai


# Load Environment Variable from the .env File.
load_dotenv()

# Retrieve API keys from environment variables.
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

print("Setup Complete: Library installed and API keys loaded")

Setup Complete: Library installed and API keys loaded


In [ ]:
import autogen
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

# We need to define an llm-config specifying the model and API key.
config_list_openai = [
    {
        "model": "gpt-4o-mini",
        "api_key": openai_api_key,
    }
]

# FIXED: Changed parentheses to curly braces to make this a valid dictionary
llm_config_openai = {
    "config_list": config_list_openai,
    "temperature": 0.7,  # Use a slightly higher temp for creative marketing ideas.
    "timeout": 120,
}

cmo_prompt = """You are the Chief Marketing Officer (CMO) of a new shoe brand (sustainable). Provide high-level
strategy, define target audiences, and guide the marketer. Focus on the big picture. Be concise."""

# FIXED: Standardized case to brand_marketer_prompt to avoid NameError
brand_marketer_prompt = """You are the Brand Marketer for the shoe brand. Brainstorm creative, specific campaign ideas
(digital, content, experiences).
Focus on tactics and details. Suggest KPIs for tour ideas."""


# Let's create Agents (Both using OpenAI Initially)
# Create the Chief Marketing Officer (CMO) Agent, using OpenAI for now.
cmo_agent_openai = autogen.ConversableAgent(
    name = "Chief_Marketing_Officer_OpenAI",
    llm_config = llm_config_openai,
    system_message = cmo_prompt,
    human_input_mode = "NEVER")

print(f"Agent '{cmo_agent_openai.name}' created (using OpenAI).")


# Create the Brand Marketer Agent, using OpenAI for now.
brand_marketer_agent_openai = autogen.ConversableAgent(
    name = "Brand_Marketer_OpenAI",
    llm_config = llm_config_openai,
    system_message = brand_marketer_prompt,
    human_input_mode = "NEVER")

print(f"Agent '{brand_marketer_agent_openai.name}' created (using OpenAI).")

Agent 'Chief_Marketing_Officer_OpenAI' created (using OpenAI).
Agent 'Brand_Marketer_OpenAI' created (using OpenAI).


### **Let's create the third agent**

In [ ]:
social_media_prompt = """You are the Social Media Strategist for a new sustainable shoe brand. Develop creative social media posts,
suggest platforms, posting times, and engagement strategies. Work with Brand Marketer and Chief Marketing Officer (CMO)."""

# Create the Social Media Strategist Agent, using OpenAI for now.
social_media_agent_openai = autogen.ConversableAgent(
    name = "Social_Media_OpenAI",
    llm_config = llm_config_openai,
    system_message = social_media_prompt,
    human_input_mode = "NEVER")

print(f"Agent '{social_media_agent_openai.name}' created (using OpenAI).")

Agent 'Social_Media_OpenAI' created (using OpenAI).


# **Test AI Agents Conversation with Similar LLM**
### Let's now make our OpenAI-only agents talk using initiate_chat() and max_turns. This shows the fundamental conversation flow before we introduce multiple models

In [ ]:
initial_task_message = """
Context: We are launching a new sustainable shoe line and need a campaign ideas.

Instruction: Brainstorm a campaign concept with specific elements.

Input: Our sustainable, futuristic shoe brand needs marketing direction.

Output: A concise campaign concept with the following structure:
Brand Marketer, let's brainstorm initial campaign ideas for our new sustainable shoe line.
Give me a distinct campaign concept.
Outline: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise. Try to arrive at a final answer.
"""

print("---Starting Agent Conversation (OpenAI only---)")
print("Chief Marketing Officer (OpenAI).Max_Turns = 4")

# Chief Marketing Officer (OpenAI) initiates the chat with Brand Marketer (OpenAI)
chat_result_openai_only = cmo_agent_openai.initiate_chat(
    recipient = brand_marketer_agent_openai, message = initial_task_message, max_turns = 4
)
print("--------------------------------------------")
print("---Conversation Ended (OpenAI Only)---")

---Starting Agent Conversation (OpenAI only---)
Chief Marketing Officer (OpenAI).Max_Turns = 4
Chief_Marketing_Officer_OpenAI (to Brand_Marketer_OpenAI):


Context: We are launching a new sustainable shoe line and need a campaign ideas.

Instruction: Brainstorm a campaign concept with specific elements.

Input: Our sustainable, futuristic shoe brand needs marketing direction.

Output: A concise campaign concept with the following structure:
Brand Marketer, let's brainstorm initial campaign ideas for our new sustainable shoe line.
Give me a distinct campaign concept.
Outline: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise. Try to arrive at a final answer.


--------------------------------------------------------------------------------
Brand_Marketer_OpenAI (to Chief_Marketing_Officer_OpenAI):

### Campaign Concept: "Step Into Tomorrow"

**Core Idea:**  
"Step Into Tomorrow" emphasizes the fusion of sustainability and futuristic design in our new shoe line.

In [ ]:
def print_chat_history(chat_result):
    """Any chat object has a chat_history attribute that contains the conversation history. This function points the conversation
    history in a readable format.
    """
    for i in chat_result.chat_history:
        print_markdown(i["name"])
        print("_" * 100)
        print_markdown(i["content"])
        print("_" * 100)

In [ ]:
# We redefine print_chat_history to use IPython's display and Markdown
# instead of the undefined print_markdown function
def print_chat_history_fixed(chat_result):
    for i in chat_result.chat_history:
        # Display the sender name in bold
        display(Markdown(f"### **{i['name']}**"))
        print("_" * 100)
        # Display the formatted markdown content
        display(Markdown(i["content"]))
        print("_" * 100)

# Let's print the chat history.
print_chat_history_fixed(chat_result_openai_only)

### **Chief_Marketing_Officer_OpenAI**

____________________________________________________________________________________________________



Context: We are launching a new sustainable shoe line and need a campaign ideas.

Instruction: Brainstorm a campaign concept with specific elements.

Input: Our sustainable, futuristic shoe brand needs marketing direction.

Output: A concise campaign concept with the following structure:
Brand Marketer, let's brainstorm initial campaign ideas for our new sustainable shoe line.
Give me a distinct campaign concept.
Outline: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise. Try to arrive at a final answer.


____________________________________________________________________________________________________


### **Brand_Marketer_OpenAI**

____________________________________________________________________________________________________


... (22 outputs left out)

### **Brand_Marketer_OpenAI**

____________________________________________________________________________________________________


### Campaign Concept: "Step Into Tomorrow"

**Core Idea:**  
"Step Into Tomorrow" aims to showcase the seamless integration of sustainability and cutting-edge design in our new shoe line. This campaign emphasizes the eco-friendly materials used in production while empowering consumers to participate in conversations about the future of fashion. By creating a digital platform that allows customers to track the environmental benefits of their purchases and share their sustainability journeys, we will foster a community dedicated to eco-conscious living.

**Target Audience:**  
Eco-conscious individuals aged 18-35, particularly urban trendsetters and fashion enthusiasts who prioritize sustainability and are active on social media. This demographic is tech-savvy and seeks brands that resonate with their values of innovation and social responsibility.

**Primary Channels:**  
1. **Social Media Campaign:**
   - **Visual Storytelling:** Leverage Instagram and TikTok for compelling visuals and video content that narrate the shoe design process, highlight sustainable materials, and feature customer stories about eco-conscious living.
   - **Augmented Reality Experiences:** Develop AR filters that allow users to virtually "try on" shoes, interact with content that shows the environmental impact of their purchase (e.g., CO2 saved, water conserved), and encourage sharing on social media.

2. **Influencer Partnerships:**
   - Collaborate with eco-conscious influencers who can authentically showcase the shoes, sharing testimonials about their sustainability features and personal eco-friendly practices.
   - Encourage influencers to engage their audience with sustainable living challenges and prompts, enhancing relatability and connection to the brand.

3. **Interactive Website Experience:**
   - Create a dedicated campaign landing page featuring a shoe customizer tool where users can select styles and visualize the environmental benefits associated with their choices.
   - Include a "Sustainability Tracker" that displays the cumulative positive impact of each purchase, reinforcing the consumer's connection to the brand and the environment.

4. **Pop-Up Experiences:**
   - Organize pop-up events in major cities where customers can try on shoes, attend workshops on sustainable living, and hear from eco-activists. These events will encourage social media sharing and community interaction, driving organic engagement.

**KPIs:**
1. **Engagement Rate:** Measure social media interactions (likes, shares, comments) and the usage of AR filters to evaluate audience engagement and interest levels.
2. **Conversion Rate:** Track the percentage of website visitors who complete a purchase, aiming for a 10% increase in online sales attributed to the campaign.
3. **User-Generated Content:** Set a goal for the number of social media posts featuring customers sharing their sustainability journeys while wearing our shoes, fostering a sense of community and encouraging brand loyalty.

This comprehensive campaign will effectively promote our sustainable shoe line while empowering consumers to join a transformative movement towards a more sustainable future, enhancing brand loyalty and driving sales.

____________________________________________________________________________________________________


## **Configure Multi-Model Agents in AutoGen using Gemini and GPT**
### The agents will now have different brains. We will configure the Chief Markrting Officer to use Google gemini model and keep the Brand Marketer using OpenAI GPT.
### We do this to showcase the ability of AutoGen to leverage multiple LLMs and because Gemini might be better at high level strategy (Chief Marketing Officer's role). While OpenAI's GPT might excel at creative brainstorming (Marketer's role). There are also cost of availability as well.

## Steps:
### 1. Define a new config_list specifically for Google Gemini models.
### 2. Create a corresponding llm_config for Gemini.
### 3. Re-create the agents, assigning the llm_config_gemini to the Chief Marketing Officer and llm_config_openai to the Brand Marketer.

In [ ]:
%pip install -U google-genai

In [ ]:

import os
from google import genai
from google.colab import userdata

GEMINI_API_KEY = "AQ.Ab8RN6KLJH2GV2h0MMF4VGYim19cMdu1SBiBAavka1yS9gWz7A"

# Set GEMINI_API_KEY in your environment variables first.
api_key = userdata.get('GEMINI_API_KEY')

if not api_key:
    raise ValueError("Please set your GEMINI_API_KEY.")

client = genai.Client(api_key=api_key)

response = client.interactions.create(
    model="gemini-3.8-flash",
    input="Explain how an AI-powered business analytics system works."
)

print(response.output_text)

An **AI-powered business analytics system** transforms raw, chaotic enterprise data into forward-looking, actionable business strategies. 

While traditional business intelligence (BI) focuses on static reports about the past (e.g., *"What were our sales last quarter?"*), an AI-driven system uses machine learning (ML), natural language processing (NLP), and automation to answer: **"Why did it happen, what will happen next, and what should we do about it?"**

Here is a step-by-step breakdown of how these systems work, from raw data to business decisions.

---

### 1. Data Ingestion & Integration (Collection)
A business generates data across dozens of disconnected tools. The AI system uses connectors and APIs to pull this data into a centralized repository (a Data Warehouse or Data Lake like Snowflake, BigQuery, or Databricks).

* **Structured Data:** Financial records, CRM entries (Salesforce), inventory databases, and website clickstreams.
* **Unstructured Data:** Customer service call

In [ ]:
import autogen
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')
gemini_api_key = userdata.get('GEMINI_API_KEY')

# Configuration for Google Gemini Agent.
# We need to define an llm-config specifying the model and API key.
config_list_gemini = [
    {
        "model": "gemini-3.8-flash",
        "api_key": google_api_key,
        "api_type": "google",

    }
]

# FIXED: Changed parentheses to curly braces to make this a valid dictionary
llm_config_gemini = {
    "config_list": config_list_gemini,
    "temperature": 0.6,  # Use a slightly randomness temp for strategic Chief Marketing Officer.
    "timeout": 120,
}

In [ ]:
# Explicitly install the missing packages needed for AutoGen's Gemini integration
import sys
import subprocess

# Run the installation unconditionally
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "ag2[gemini]", "google-genai", "vertexai"])

# Completely purge autogen and its modules from memory to force a clean check
modules_to_remove = [name for name in sys.modules if name.startswith("autogen")]
for module_name in modules_to_remove:
    del sys.modules[module_name]

# Invalidate import caches
import site
import importlib
for site_dir in site.getsitepackages():
    if site_dir not in sys.path:
        sys.path.append(site_dir)
importlib.invalidate_caches()

# Clean re-import
import autogen

# Let's create our agents using different models.
# Create the Chief Marketing Officer Agent now using Google Gemini.
cmo_agent_gemini = autogen.ConversableAgent(
    name = "Chief_Marketing_Officer_gemini",
    llm_config = llm_config_gemini,
    system_message = cmo_prompt,
    human_input_mode = "NEVER")

# Create the Brand Marketer Agent using OpenAI'GPT.
# We can re-use the llm_config_openai defined earlier.
brand_marketer_agent_openai_mixed = autogen.ConversableAgent(
    name = "Brand_Marketer_OpenAI",
    llm_config = llm_config_openai,
    system_message = brand_marketer_prompt,
    human_input_mode = "NEVER")

print(f"Agent '{cmo_agent_gemini.name}' created (using Google Gemini).")
print(f"Agent '{brand_marketer_agent_openai_mixed.name}' created (using OpenAI).")

Agent 'Chief_Marketing_Officer_gemini' created (using Google Gemini).
Agent 'Brand_Marketer_OpenAI' created (using OpenAI).


### **Let's now use CLAUDE for "Social Media Strategist"**

In [ ]:
import autogen
from google.colab import userdata
claude_api_key = userdata.get('CLAUDE_API_KEY')

config_list_claude = [
    {
        "model": "claude-3.5-sonnet-20240620",
        "api_key": claude_api_key,
        "api_type": "anthropic",
    }
]

# FIXED: Set temperature <= 1.0 directly in the config
llm_config_claude = {
    "config_list": config_list_claude,
    "temperature": 0.6,
    "timeout": 120,
}

In [ ]:
# Ensure the anthropic package is installed for Claude support in AutoGen
import sys
import subprocess

# Force run pip installation of anthropic
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "anthropic"])

# Purge autogen from sys.modules to clear cached import failures
modules_to_remove = [name for name in sys.modules if name.startswith("autogen")]
for module_name in modules_to_remove:
    del sys.modules[module_name]

# Invalidate import caches
import site
import importlib
for site_dir in site.getsitepackages():
    if site_dir not in sys.path:
        sys.path.append(site_dir)
importlib.invalidate_caches()

# Re-import autogen now that anthropic is installed
import autogen

# Adjust the temperature in llm_config_claude so it is <= 1.0 to satisfy the validation constraint
llm_config_claude["temperature"] = 0.7

social_media_prompt = """You are the Social Media Strategist for a new sustainable shoe brand. Develop creative social media posts,
suggest platforms, posting times, and engagement strategies. Work with Brand Marketer and Chief Marketing Officer (CMO)."""

# Create the Social Media Strategist Agent, using Claude.
social_media_agent_claude = autogen.ConversableAgent(
    name = "Social_Media_Claude",
    llm_config = llm_config_claude,
    system_message = social_media_prompt,
    human_input_mode = "NEVER")

print(f"Agent '{social_media_agent_claude.name}' created (using Claude).")

Agent 'Social_Media_Claude' created (using Claude).


### Let's now trigger AI Agents Conversations among gemini-powered Chief Marketing Officer (CMO), OpenAI GPT-powered Brand Marketer (A Multi-Model Team).

In [ ]:
%pip install -U google-genai

In [ ]:
print("---Starting Agent Conversation (Multi-Model: Gemini + OpenAI---)")
print("Chief Marketing Officer (Gemini) initiating chat with Brand Marketer (OpenAI). Max_Turns = 4")
print("---------------------------------------------------------------------------")

# 3. Initiate the chat with the OpenAI Brand Marketer
chat_result_multi_model = cmo_agent_gemini.initiate_chat(
    recipient = brand_marketer_agent_openai_mixed,
    message = initial_task_message,
    max_turns=4)

print("--------------------------------------------")
print("---Conversation Ended (Multi-Model)---")

---Starting Agent Conversation (Multi-Model: Gemini + OpenAI---)
Chief Marketing Officer (Gemini) initiating chat with Brand Marketer (OpenAI). Max_Turns = 4
---------------------------------------------------------------------------
Chief_Marketing_Officer_gemini (to Brand_Marketer_OpenAI):


Context: We are launching a new sustainable shoe line and need a campaign ideas.

Instruction: Brainstorm a campaign concept with specific elements.

Input: Our sustainable, futuristic shoe brand needs marketing direction.

Output: A concise campaign concept with the following structure:
Brand Marketer, let's brainstorm initial campaign ideas for our new sustainable shoe line.
Give me a distinct campaign concept.
Outline: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise. Try to arrive at a final answer.


--------------------------------------------------------------------------------
Brand_Marketer_OpenAI (to Chief_Marketing_Officer_gemini):

**Campaign Concept: "Step 

ValueError: Unable to find your project. Please provide a project ID by:
- Passing a constructor argument
- Using vertexai.init()
- Setting project using 'gcloud config set project my-project'
- Setting a GCP environment variable
- To create a Google Cloud project, please follow guidance at https://developers.google.com/workspace/guides/create-project

In [ ]:
print_chat_history_fixed(chat_result_multi_model)

### **Adding Human Guidance (User Proxy Agent) & Leveraging Group Chat**
### Now, let's add the UserProxyAgent back, allowing you to interact with the multi-model team (Gemini CMO + OpenAI Marketer)

In [ ]:
# Create the User Proxy Agent (Represents you)
user_proxy_agent = autogen.UserProxyAgent(
    name = "Human_User_Proxy",
    human_input_mode = "Always",
    max_consecutive_auto_reply = 1,
    is_termination_msg = lambda x: x.get("content", "").rstrip().lower() in ["exit", "quit", "terminate"],
    code_execution_config = False,
    system_message = "You are the human user interacting with a multi-model AI team (Gemini CMO, OpenAI Marketer)."
    "Guide the brainstorm. Type 'exit' to end.",
)
print(f"Agent '{user_proxy_agent.name}' created for HIL with multi-model team.")

# Initializing HIL chat with the multi-model team.
print("---Starting Human-in-the-Loop Conversation (Multi-Model)---")
print("You will interact with Gemini CMO and OpenAI Marketer. Type 'exit' to end.")
print("-------------------------------------------------------------------------")

In [ ]:
# Reset Agents for a clean new session.
cmo_agent_gemini.reset()
brand_marketer_agent_openai_mixed.reset()
social_media_agent_claude.reset()
user_proxy_agent.reset()

In [ ]:
from autogen import GroupChat, GroupChatManager

# Create a group chat with multiple agents.
# This sets up a collaborative chat environment where multiple agents can interact.
groupchat = GroupChat(
    agents=[cmo_agent_gemini, brand_marketer_agent_openai_mixed, user_proxy_agent, social_media_agent_claude],  # List of agents participating in the group chat.
    messages=[],
    max_round=20,
    )
# Create a manager for the group chat.
# The GroupChatManager orchestrates the conversation flow between agents.
# It determines which agent should speak next and handles the overall conversation logic.
group_manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config_openai)  # Uses OpenAI's LLM to manage the conversation

In [ ]:
# User Proxy initiates the chat-Let's give a new task.
group_chat_result = group_manager.initiate_chat(
    recipient = user_proxy_agent,
    message """Let's develop a Tik Tok campaign for our targeting Gen Z.
    CMO (Gemini): Please outline the strategic goals.
    Brand Marketer (OpenAI): Suggest a campaign hook.
    Social Media Strategist (Claude): Provide a Social Media Plan.""",
    max_round=20
)
print("---------------------------------------------------------")
print("---Conversation Ended (Human terminated or Max Turns)---")

In [ ]:
print_chat_history(group_chat_result)